# One system, two jurisdictions

> **Demonstration only:** frozen synthetic data, not evidence about any real decision. Reasonsmith reports evidence and refusals; it does not certify compliance or provide legal advice.

The first table is the payoff: the same shipped credit system is checked against the US `ecoa` pack and the EU `gdpr` and `eu_ai_act` packs. A verdict can diverge because a pack asks for a different signal or because its applicability gate is different; the table keeps that reason beside the result.

In [1]:
import html

from IPython.display import HTML, display

from reasonsmith.demo import deployed_credit_system
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack


def show_table(rows, columns, title):
    head = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '—')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"<h3>{html.escape(title)}</h3><table><thead><tr>{head}</tr></thead>"
                      f"<tbody>{body}</tbody></table>"))

def difference_reason(requirement, result):
    outcome = result["outcome"]
    if outcome == "not_applicable":
        return f"Pack scope {requirement.scope!r} is not reached: no regulatory class was declared."
    if outcome == "unattainable":
        missing = ", ".join(result.get("signals_missing", []))
        return f"The duty requires {missing}; this system does not expose it."
    text = requirement.rationale if outcome == "satisfied" else result["evidence_summary"]
    return text.split(". ", 1)[0].rstrip(".") + "."

system = deployed_credit_system()
rows = []
for jurisdiction, pack_name in [("US", "ecoa"), ("EU", "gdpr"), ("EU", "eu_ai_act")]:
    pack = load_pack(pack_name)
    report = check_conformance(system, pack)
    for requirement, result in zip(pack.requirements, report.to_dict()["results"], strict=True):
        rows.append({"Duty": f"{requirement.id} ({requirement.article_clause})",
                     "Jurisdiction": jurisdiction, "Verdict": result["verdict"],
                     "Rung": result["strength"] or "—",
                     "Why this landed here": difference_reason(requirement, result)})
headline = ("ecoa_reg_b_1002_9_b_2_principal_reasons_complete",
            "gdpr_art22_3_safeguards_human_intervention",
            "gdpr_art22_1_automated_decision_prohibition",
            "eu_ai_act_art86_1_main_elements_of_the_decision")
rows.sort(key=lambda row: next((i for i, p in enumerate(headline) if row["Duty"].startswith(p)), 4))
show_table(rows, ["Duty", "Jurisdiction", "Verdict", "Rung", "Why this landed here"],
           "One frozen system across US and EU packs")


Duty,Jurisdiction,Verdict,Rung,Why this landed here
ecoa_reg_b_1002_9_b_2_principal_reasons_complete (12 CFR 1002.9(b)(2)),US,violated,probed,Violated on 1 of 2 certified decision(s): the stated reasons are not all the reasons.
gdpr_art22_3_safeguards_human_intervention (Article 22(3)),EU,satisfied,observed,Safeguards providing contestation endpoints and human review pathways.
gdpr_art22_1_automated_decision_prohibition (Article 22(1)),EU,inconclusive,unattainable,The duty requires provenance_active_exceptions; this system does not expose it.
eu_ai_act_art86_1_main_elements_of_the_decision (Article 86(1)),EU,not_applicable,—,Pack scope 'high-risk' is not reached: no regulatory class was declared.
ecoa_reg_b_1002_9_a_1_timing_of_notice (12 CFR 1002.9(a)(1)),US,satisfied,observed,"Every decision the log records was notified within the deadline its own paragraph sets: 30 days under (i)-(iii), or 90 days where the record says the applicant did not accept a counteroffer, which is the only case (iv) reaches."
ecoa_reg_b_1002_9_a_2_written_statement (12 CFR 1002.9(a)(2)),US,satisfied,observed,"Every decision the log records carries the decision record and the model version that produced it, together with one of the two contents the clause accepts: a statement of specific reasons under point (i), or a disclosure of the applicant's right to request one under point (ii)."
ecoa_reg_b_1002_9_b_2_specific_reasons (12 CFR 1002.9(b)(2)),US,satisfied,observed,"Where a decision carries a statement of reasons — which is when this clause bites, because by its own words it governs the statement paragraph (a)(2)(i) requires, and a creditor that lawfully took the (a)(2)(ii) disclosure branch has none yet — that statement names the model version and the scope it speaks for, and is not one of the two the clause itself calls insufficient: that the action rested on the creditor's internal standards or policies, or that the applicant failed to achieve a qualifying score."
ecoa_reg_b_1002_9_c_2_incompleteness_notice_runs_out (12 CFR 1002.9(c)(2)),US,inconclusive,unattainable,The duty requires artifact_logs_incompleteness_notice_sent; this system does not expose it.
ecoa_reg_b_1002_4_a_no_disparate_treatment (12 CFR 1002.4(a)),US,inconclusive,—,"Not evaluated: the system exposes no decide(), so there is no twin decision to run."
gdpr_art22_1_no_prohibited_decision_for_any_input (Article 22(1)),EU,inconclusive,unattainable,"The duty requires artifact_logs_human_intervention_route, artifact_logs_significant_effect, artifact_logs_solely_automated, provenance_basis_contract, provenance_basis_explicit_consent, provenance_basis_union_or_member_state_law; this system does not expose it."


The divergence is substantive where the pack asks a different question: the ECOA and EU AI Act reason-completeness duties both use the exposed inference artefact, while GDPR's duties ask for signals this system does not emit. The EU AI Act rows are **not applicable**, not passes or failures: its shipped requirements are scoped to `high-risk`, and this system declares no regulatory class. That class is never inferred.

The legal wording and formalised limits come from the shipped pack files (`src/reasonsmith/packs/ecoa.toml`, `gdpr.toml`, and `eu_ai_act.toml`), whose source record is [`docs/legal-sources.md`](../docs/legal-sources.md). The applicability and evidence semantics are documented in [`docs/semantics.md`](../docs/semantics.md).